# **LLM Pipeline to Process Extracted Dataset (Phase 1)**

In [ ]:
!pip install pandas

### **Start custom worker on MultiCare dataset**

In [ ]:
from DatasetWorker import mount_worker
from DatasetWorkerHelpers import get_secret

DATASET_NAME = "extracted_dataset_1792212375"
worker = mount_worker(DATASET_NAME)

### **Clean up junk files (safe to run everytime)**

In [2]:
worker.clean_dataset()

### **Eliminate cases with ambiguous demographic**

In [ ]:
worker.delete_unknown_demographics()

### **Generate .txt queries for every cases.csv found**

In [ ]:
worker.generate_diagnosis_questions()

### **Attach scores to cases (requiring prompt 3 and result.json)**

In [ ]:
worker.attach_scores()

### **Show dataset quality metrics**

In [ ]:
worker.display_metrics()

# **Final Processing (Split into train and test)**

### **Build big dataset (Generate `normalized_diseases.csv` and `evaluated_cases.csv`)**

In [ ]:
worker.build_master_dataset()

### **Split into train and test. Mask test set (use prompt 4)**

In [7]:
train_df, test_df = worker.assign_splits()

masked_df = worker.get_masked_test_cases()

Another path: Structured cases

In [8]:
train_df, test_df, masked_df = worker.read_soap_notes()

,case_id,pair_id,split,target_disease,gender,age,raw_case_text,soap_note
0,PMC9233216_01,PMC9233216_01::Brucellosis,train,Brucellosis,Female,11,"The patient is an 11-year-old Lebanese girl, d...",{'subjective': {'chief_complaint': 'An 11-year...
1,PMC9353035_01,PMC9353035_01::Brucellosis,train,Brucellosis,Male,40,A 40-year-old male farmer without any underlyi...,{'subjective': {'chief_complaint': 'A 40-year-...
2,PMC8710768_01,PMC8710768_01::Brucellosis,train,Brucellosis,Male,67,"A 67-year-old man, a local farmer specializing...",{'subjective': {'chief_complaint': 'A 67-year-...
3,PMC9389076_01,PMC9389076_01::Brucellosis,train,Brucellosis,Male,42,A 42-year-old man was admitted to our hospital...,{'subjective': {'chief_complaint': 'A 42-year-...
4,PMC4154277_01,PMC4154277_01::Brucellosis,train,Brucellosis,Female,25,A 25 year old woman was referred with a histor...,{'subjective': {'chief_complaint': 'A 25-year-...
...,...,...,...,...,...,...,...,...
440,PMC10847898_01,PMC10847898_01::Q Fever,train,Q Fever,Male,39,We present a clinical case of a 39-year-old ma...,{'subjective': {'chief_complaint': 'A 39-year-...
441,PMC12668970_01,PMC12668970_01::Q Fever,train,Q Fever,Female,50,"A 50-year-old woman was admitted on March 29, ...",{'subjective': {'chief_complaint': 'A 50-year-...
442,PMC12739083_01,PMC12739083_01::Q Fever,train,Q Fever,Female,73,A 73-year-old female with a medical history of...,{'subjective': {'chief_complaint': 'A 73-year-...
443,PMC12289594_01,PMC12289594_01::Q Fever,train,Q Fever,Male,66,"A 66-year-old male was admitted on January 15,...",{'subjective': {'chief_complaint': 'A 66-year-...


### **Upload to HuggingFace**

In [ ]:
from huggingface_hub import login, whoami

# Retrieve token and set env var in one line
hf_token = get_secret("HF_TOKEN")

login(token=hf_token)
print(whoami())

In [11]:
from DatasetWorkerHelpers import build_hf_dataset

dataset_dict = build_hf_dataset(train_df=train_df, test_df=test_df, masked_df=masked_df)
REPO_ID = "choisongjun/Tropical_Diseases_Repo_v3.2"

In [ ]:
dataset_dict.push_to_hub(REPO_ID, token=hf_token)
print(f"\nPushed to https://huggingface.co/datasets/{REPO_ID}")